# Expected goals -- distance + angle

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
sys.path.append(str(PROJECT_ROOT))

In [2]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    log_loss,
    brier_score_loss,
    roc_auc_score
)

from src.data.split_data import train_test_split_by_match

/opt/anaconda3/lib/python3.12/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.12/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


In [15]:
df = pd.read_csv("../../data/processed/shots.csv")

df.head()

,match_id,period,minute,second,team,player,x,y,goal,body_part,...,first_time,aerial_won,one_on_one,deflected,open_goal,saved_off_target,redirect,saved_to_post,follows_dribble,statsbomb_xg
0,3754300,1,12,14,Watford,Troy Deeney,108.9,40.6,0,Right Foot,...,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.085717
1,3754300,1,12,52,Watford,Troy Deeney,94.7,34.4,0,Right Foot,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.023742
2,3754300,1,12,58,Watford,Miguel Arturo Layún Prado,107.4,41.7,1,Right Foot,...,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.148063
3,3754300,1,18,16,Everton,Gareth Barry,111.8,40.2,0,Head,...,NaN,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.074549
4,3754300,1,23,14,Everton,Ross Barkley,96.5,39.7,0,Right Foot,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.036574


## Model 2 -- shot distance & shot angle

### Shot Distance

For a shot at coordinates $(x_i,y_i)$, the distance to the center of the
goal is:

$$
d_i =
\sqrt{(120-x_i)^2 + (40-y_i)^2}
$$

In [16]:
df["distance"] = np.sqrt(
    (120 - df["x"])**2 +
    (40 - df["y"])**2
)

### Shot Angle

The angle of a shot is defined as the angle subtended by the two goalposts
from the shot location.

The two goalposts are at $(120,36)$ and $(120,44)$.

For a shot at $(x_i,y_i)$, define the vectors to the two posts as:

$$
\mathbf{a}_i =
(120-x_i,\;36-y_i)
$$

$$
\mathbf{b}_i =
(120-x_i,\;44-y_i)
$$

Using the dot product, the angle is:

$$
\theta_i =
\cos^{-1}
\left(
\frac{\mathbf{a}_i\cdot\mathbf{b}_i}
{\|\mathbf{a}_i\|\|\mathbf{b}_i\|}
\right)
$$

In [17]:
goalpost_near = np.array([120, 36])
goalpost_far = np.array([120, 44])

shot_locations = df[["x", "y"]].values

vector_a = goalpost_near - shot_locations
vector_b = goalpost_far - shot_locations

dot_product = np.sum(vector_a * vector_b, axis=1)

magnitude_a = np.linalg.norm(vector_a, axis=1)
magnitude_b = np.linalg.norm(vector_b, axis=1)

cos_angle = dot_product / (magnitude_a * magnitude_b)

# Protect against tiny floating-point errors
cos_angle = np.clip(cos_angle, -1, 1)

df["angle"] = np.arccos(cos_angle)

In [18]:
df["angle"].describe()

count    9908.000000
mean        0.443055
std         0.274477
min         0.001063
25%         0.267127
50%         0.341150
75%         0.539819
max         2.787532
Name: angle, dtype: float64

In [19]:
df["angle_degrees"] = np.degrees(df["angle"])

In [20]:
df[["x", "y", "distance", "angle_degrees"]].head(10)

,x,y,distance,angle_degrees
0,108.9,40.6,11.116204,39.539979
1,94.7,34.4,25.912352,17.160452
2,107.4,41.7,12.714165,34.685947
3,111.8,40.2,8.202439,51.985000
4,96.5,39.7,23.501915,19.316784
5,91.3,41.8,28.756391,15.808500
6,104.7,19.7,25.420071,10.991713
7,93.7,49.3,27.895878,15.432182
8,100.0,23.1,26.184156,13.438568
9,116.5,33.1,7.736924,32.553953


In [21]:
df[["distance", "angle_degrees"]].describe()

,distance,angle_degrees
count,9908.000000,9908.000000
mean,19.071931,25.385163
std,8.291229,15.726347
min,0.921954,0.060911
25%,12.126417,15.305260
50%,18.791620,19.546465
75%,25.339840,30.929350
max,86.137100,159.713824


The logistic regression model is:

$$
P(Y_i=1 \mid d_i,\theta_i)
=
\frac{1}
{1+\exp\left(
-(\beta_0+\beta_1d_i+\beta_2\theta_i)
\right)}
$$

where $d_i$ is shot distance and $\theta_i$ is shot angle.

In [22]:
train, test = train_test_split_by_match(
    df,
    test_size=0.2,
    random_state=42
)

print(f"Training matches: {train['match_id'].nunique()}")
print(f"Test matches:     {test['match_id'].nunique()}")

Training matches: 304
Test matches:     76


In [23]:
X_train = train[["distance", "angle"]]
y_train = train["goal"]

X_test = test[["distance", "angle"]]
y_test = test["goal"]

model_distance_angle = LogisticRegression()

model_distance_angle.fit(
    X_train,
    y_train
)

LogisticRegression()

In [24]:
print(f"Intercept:          {model_distance_angle.intercept_[0]:.4f}")
print(f"Distance coefficient: {model_distance_angle.coef_[0][0]:.4f}")
print(f"Angle coefficient:    {model_distance_angle.coef_[0][1]:.4f}")

Intercept:          -1.2989
Distance coefficient: -0.0821
Angle coefficient:    0.9228


In [25]:
test["xg_distance_angle"] = model_distance_angle.predict_proba(
    X_test
)[:, 1]

In [26]:
logloss_distance_angle = log_loss(
    y_test,
    test["xg_distance_angle"]
)

brier_distance_angle = brier_score_loss(
    y_test,
    test["xg_distance_angle"]
)

auc_distance_angle = roc_auc_score(
    y_test,
    test["xg_distance_angle"]
)

print(f"Log Loss:    {logloss_distance_angle:.4f}")
print(f"Brier Score: {brier_distance_angle:.4f}")
print(f"ROC AUC:     {auc_distance_angle:.4f}")

Log Loss:    0.2660
Brier Score: 0.0746
ROC AUC:     0.7473


## Model Comparison

| Model | Log Loss | Brier Score | ROC AUC |
|---|---:|---:|---:|
| Constant probability | 0.2967 | 0.0796 | 0.5000 |
| Distance | 0.2667 | 0.0750 | 0.7398 |
| Distance + Angle | 0.2660 | 0.0746 | 0.7473 |

Model 2 adds shot angle to distance. This produces a further, although relatively small, improvement across all three metrics. The ROC AUC increases from 0.7398 to 0.7473, while both Log Loss and Brier Score decrease, indicating slightly better predictive performance and calibration.